In [1]:
%pip install -q ipywidgets


[notice] A new release of pip is available: 26.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

STATE_PATH = Path("decision_probe_results/demo_state.pkl")
assert STATE_PATH.exists(), f"Saved results not found. Current folder: {Path.cwd()}"
with STATE_PATH.open("rb") as file:
    state = pickle.load(file)
globals().update(state)

print(f"Loaded {len(probe_results):,} candidate edits and {len(stability_summary)} stability results.")

Loaded 1,485 candidate edits and 29 stability results.


In [3]:
display(HTML("<h2>TabPFN Decision Probe</h2><p>What would flip this prediction—and does the flip survive training-sample changes?</p>"))

options = [(f"Record {rid} | original risk {baseline.at[rid, 'p_bad']:.1%}", rid) for rid in summary.index]
selector = widgets.Dropdown(options=options, description="Record:", layout=widgets.Layout(width="500px"))
default_id = 261 if 261 in summary.index else summary.index[0]
selector.value = default_id
fragile_button = widgets.Button(description="Example: record 261")
stable_button = widgets.Button(description="Example: record 840")
output = widgets.Output()

def inspect_record(change=None):
    with output:
        clear_output(wait=True)
        rid = selector.value
        print(f"RECORD {rid}")
        print(f"Original bad-credit probability: {baseline.at[rid, 'p_bad']:.1%} | Threshold: {THRESHOLD:.0%}")
        if rid in best_flips.index:
            edit, stats = best_flips.loc[rid], stability_summary.loc[rid]
            print(f"Smallest tested edit: {edit.feature} {edit.old_value:g} → {edit.new_value:g}")
            print(f"Relative reduction: {edit.relative_reduction:.1%} | Modified probability: {edit.modified_p_bad:.1%}")
            print(f"Paired flips retained: {int(stats.retained_flip_count)}/{int(stats.repeats)} training subsets")
            print(f"Original remained high risk: {int(stats.original_high_risk_count)}/{int(stats.repeats)}")
        else:
            print("No single-feature flip found within the tested reductions.")

        fig, axes = plt.subplots(1, 2, figsize=(11, 4))
        for ax, feature in zip(axes, FEATURES):
            curve = probe_results.loc[(probe_results.record_id == rid) & (probe_results.feature == feature)]
            curve = curve[["new_value", "modified_p_bad"]].rename(columns={"new_value": "value", "modified_p_bad": "probability"})
            original = pd.DataFrame({"value": [X_test.at[rid, feature]], "probability": [baseline.at[rid, "p_bad"]]})
            curve = pd.concat([curve, original]).sort_values("value")
            ax.plot(curve.value, curve.probability, "o-", color="#379f87", markersize=4, label="Tested values")
            ax.scatter(original.value, original.probability, color="#d66b58", s=70, zorder=3, label="Original")
            ax.axhline(THRESHOLD, color="black", linestyle="--", label="Threshold")
            ax.set(xlabel=feature.replace("_", " ").title(), ylabel="Predicted bad-credit probability", ylim=(0, 1))
            ax.legend(fontsize=8)
        plt.tight_layout()
        plt.show()

        if rid in stability_summary.index:
            runs = stability_runs.loc[stability_runs.record_id == rid].sort_values("repeat")
            fig, ax = plt.subplots(figsize=(9, 3.5))
            ax.plot(runs["repeat"], runs.original_p_bad, "o-", color="#d66b58", label="Original record")
            ax.plot(runs["repeat"], runs.modified_p_bad, "o-", color="#379f87", label="Fixed edit")
            ax.axhline(THRESHOLD, color="black", linestyle="--", label="Threshold")
            ax.set(xlabel="Training subset", ylabel="Predicted bad-credit probability", ylim=(0, 1),
                   xticks=runs["repeat"], title="Does the same edit retain its flip?")
            ax.legend(fontsize=8)
            plt.tight_layout()
            plt.show()

        print("Hypothetical model sensitivity only. Minimum means smallest tested relative reduction.")
        print("Feature combinations may be unrealistic; these edits do not establish feasible financial actions.")
        print("Stability describes the tested subsets and does not guarantee future outcomes.")

def select_example(rid):
    if rid in summary.index:
        selector.value = rid

fragile_button.on_click(lambda _: select_example(261))
stable_button.on_click(lambda _: select_example(840))
selector.observe(inspect_record, names="value")
display(widgets.VBox([selector, widgets.HBox([fragile_button, stable_button]), output]))
inspect_record()

In [4]:
display(HTML("<h3>Experiment summary</h3>"))
overview = pd.DataFrame({
    "Measure": ["Held-out test records", "Predicted high-risk records", "Candidate edits tested",
                "Records with a tested flip", "Edits retained in every tested subset"],
    "Value": [len(X_test), len(summary), len(probe_results), len(best_flips),
              int((stability_summary.retained_flip_count == stability_summary.repeats).sum())]
})
display(overview)
display(HTML("<h3>Synthetic ground-truth validation</h3>"))
display(effect_metrics.round(4))
print("Sign agreement excludes true probability changes with magnitude ≤ 0.01.")
print("Synthetic results describe three seeds and the specified data-generating process.")

,Measure,Value
0,Held-out test records,200
1,Predicted high-risk records,45
2,Candidate edits tested,1485
3,Records with a tested flip,29
4,Edits retained in every tested subset,1


,tested_edits,effect_MAE,effect_RMSE,mean_absolute_model_delta,sign_agreement
correlated_proxy,474.0,0.0147,0.0236,0.0147,NaN
independent_noise,437.0,0.0099,0.0187,0.0099,NaN
signal_1,429.0,0.0218,0.0311,0.0819,0.9975
signal_2,435.0,0.0200,0.0309,0.0737,0.9720


Sign agreement excludes true probability changes with magnitude ≤ 0.01.
Synthetic results describe three seeds and the specified data-generating process.


In [6]:
import inspect
from importlib.metadata import version
from tabpfn_client import TabPFNClassifier

print("tabpfn-client version:", version("tabpfn-client"))
print("Classifier constructor:", inspect.signature(TabPFNClassifier))

tabpfn-client version: 0.6.1
Classifier constructor: (model_path: 'str | None' = None, n_estimators: 'int | None' = None, softmax_temperature: 'float | None' = None, balance_probabilities: 'bool' = False, average_before_softmax: 'bool | None' = None, ignore_pretraining_limits: 'bool' = False, inference_precision: "Literal['autocast', 'auto'] | None" = None, random_state: 'int | None' = 0, inference_config: 'dict[str, Any] | None' = None, categorical_features_indices: 'list[int] | None' = None, fit_mode: 'FitModeLiteral | None' = None, paper_version: 'bool' = False, thinking_mode: 'bool' = False, thinking_effort: 'ThinkingEffort | None' = None, thinking_timeout_s: 'float | None' = None, thinking_metric: 'str | None' = None, group_col: 'str | list[str] | None' = None, time_col: 'str | None' = None, group_time_col: 'str | None' = None, api_mode: 'ApiMode' = <ApiMode.AUTO: 'auto'>, client_options: 'ClientOptions | None' = None, text_handling: 'TextHandling' = 'advanced')


In [7]:
import inspect
from tabpfn_client import TabPFNClassifier

source = inspect.getsource(TabPFNClassifier).splitlines()
matches = [i for i, line in enumerate(source) if any(term in line.lower() for term in ["model_path", "default_model", "3.5"])]
shown = set()
for i in matches[:12]:
    for j in range(max(0, i - 2), min(len(source), i + 4)):
        if j not in shown:
            print(source[j])
            shown.add(j)
    print()

        # to parse model names by substring, so "v2.5_default-2" must precede "v2.5_default".
        "v2.5_default-2",
        *_DEFAULT_MODEL_NAMES,
        "v2.5_large-features-L",
        "v2.5_large-features-XL",
        "v2.5_large-samples",

        self,
        # start: tabpfn_config
        model_path: str | None = None,
        n_estimators: int | None = None,
        softmax_temperature: float | None = None,
        balance_probabilities: bool = False,

        This constructs a classifier using the latest model and settings. If you would
        like to use a previous model version, use `create_default_for_version()`
        instead. You can also use `model_path` to specify a particular model

        Parameters
        ----------

        model_path: str, default="auto"
            The name of the model to use. "auto" lets the server pick the
            latest default model; "default" is accepted as a backward-compatible
            alias. Use `create_default_for_version

In [8]:
import inspect
from tabpfn_client import TabPFNClassifier

print("Version-selection signature:")
print(inspect.signature(TabPFNClassifier.create_default_for_version))
print("\nImplementation:")
print(inspect.getsource(TabPFNClassifier.create_default_for_version))

Version-selection signature:
(version: 'ModelVersion | str', **overrides) -> 'Self'

Implementation:
    @classmethod
    def create_default_for_version(
        cls, version: ModelVersion | str, **overrides
    ) -> Self:
        """Construct an estimator that uses the given version of the model.

        Any kwargs will override the default settings, except for `model_path`.
        """
        try:
            version = ModelVersion(version)
        except ValueError:
            raise ValueError(
                f"Invalid model version: {version}. "
                f"Available versions are: {', '.join(list(ModelVersion))}."
            )
        options = overrides.copy()
        options["model_path"] = f"{version.value}_default"
        return cls(**options)



In [9]:
from tabpfn_client import TabPFNClassifier

ModelVersion = TabPFNClassifier.create_default_for_version.__func__.__globals__["ModelVersion"]
available_versions = [item.value for item in ModelVersion]
print("Available versions:", available_versions)

MODEL_VERSION = next((v for v in available_versions if v in ("v3.5", "3.5")), None)
assert MODEL_VERSION is not None, "Share the available versions above so we can select the correct identifier."

def make_model():
    return TabPFNClassifier.create_default_for_version(MODEL_VERSION, random_state=SEED)

print("Pinned model:", make_model().model_path)

Available versions: ['v2', 'v2.5', 'v2.6', 'v3', 'v3.5', 'v3.5-fast']
Pinned model: v3.5_default
